In [0]:
import pyspark
from pyspark import SparkContext

In [0]:
# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email = dbutils.secrets.get(scope="gcp_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="gcp_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="gcp_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="gcp_scope", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = spark._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
hubstat_filepath = "gs://modelsonpaper/hub_stats/models.parquet"
hubstat_filepathrdd = spark.read.parquet(hubstat_filepath).rdd

In [0]:
hubstat_filepathrdd.take(1)

In [0]:
hubstat_rows = hubstat_filepathrdd.map(lambda r: (r.id, r.downloads, r.tags, r.pipeline_tag, r.createdAt))
hubstat_rows.take(1)

In [0]:
headers= ["id", "downloads", "tags", "pipeline_tag", "createdAt"]

In [0]:
def filter_tags (rdd, tag: str):
    return rdd.filter(lambda x: [i for i in x[2] if i[:6] == tag])

entry_with_arxiv = filter_tags(hubstat_rows, "arxiv:")

In [0]:
prepro_hub = entry_with_arxiv.map(lambda x: (x[0], x[1], [i[6:] for i in x[2] if i[:6] == "arxiv:"], x[3], x[4]))


In [0]:
prepro_hub.take(3)

In [0]:
cleaned_hub = prepro_hub.map(lambda x: "\t".join([x[0],str(x[1]),",".join(x[2]),x[3] if x[3] is not None else "none",str(x[4])]))
cleaned_hub.take(3)

In [0]:
cleaned_hub.saveAsTextFile("gs://modelsonpaper/processed/hub_stats")

In [0]:
cleaned_hub.count()

In [0]:

sc.textFile("gs://modelsonpaper/processed/hub_stats").count()